# AIME — Video Generation with Free/Open-Weight T2V Models

Generates videos from the 57 video-compatible AIME prompts (`Modality` in `{'Video','Both'}`) using one of **four
free, non-API models** (no payment, no API key) via HuggingFace `diffusers`:

| MODEL_KEY | Model | License | Fits 8GB local? |
|---|---|---|---|
| `wan21` | Wan2.1-T2V-1.3B | Apache 2.0 | Yes (with offload) |
| `cogvideox` | CogVideoX-2B | Apache 2.0 | Yes (with offload) |
| `ltxvideo` | LTX-Video-0.9.7-distilled | Research-use license | Yes, fast (4-10 steps) |
| `hunyuanvideo` | HunyuanVideo (13B) | Open, community license | **No** - needs cloud GPU (~14GB even int4) |

Set `MODEL_KEY` in the config cell to choose which model to run - mirrors the `MODEL_KEY` pattern in
`generate_SD35_aime.ipynb`. Kling and Seedance (paid APIs) are intentionally NOT in this notebook - see
`generate_kling.ipynb` / `generate_seedance.ipynb` for those.

**Output structure:**
```
dataset/media/videos/{ModelName}/
    P001_amb_<hash>.mp4
    ...
```
A CSV index is written to `dataset/labels/generated_{ModelName}.csv`.

**Prerequisites:**
```bash
pip install torch diffusers>=0.37 transformers accelerate ftfy imageio imageio-ffmpeg
# HunyuanVideo quantized path additionally needs:
pip install bitsandbytes
```

## Hardware note - READ BEFORE RUNNING

The local dev GPU is an **RTX 4070 Laptop with 8GB VRAM**.

- **Wan2.1-T2V-1.3B** ("Consumer-Grade Efficiency" variant per the Wan model card) needs ~8.19GB VRAM stock, and fits
  on this GPU with `enable_model_cpu_offload()` + VAE in fp32/tiling at 480p. **Not** the 14B variant.
- **CogVideoX-2B** is the smallest CogVideoX checkpoint (2B params) - fits with offload + tiling, no quantization
  needed. Best-documented `diffusers` API of the free options.
- **LTX-Video-0.9.7-distilled** is guidance/timestep-distilled for speed: `guidance_scale` must be `1.0` and
  `num_inference_steps` should be 4-10 (higher raises no quality benefit on the distilled checkpoint, only cost).
  Fastest of the four options.
- **HunyuanVideo** is a 13B-parameter model. Per the official diffusers docs, *even the int4-quantized checkpoint with
  model-offloading and VAE tiling needs ~14GB VRAM* - this will almost certainly **OOM on this 8GB card**. The
  loading/generation code below is implemented correctly and will run as-is on a GPU with enough headroom (cloud /
  rented GPU, e.g. A10/A100/L40S), but do not expect it to complete locally. Treat any local HunyuanVideo run here as
  "verify the code doesn't crash before the OOM", not as a way to actually get videos out of this machine.

`SMOKE_TEST = True` below restricts the loop to a single prompt and low resolution/frame count so you can sanity-check
the pipeline end-to-end without committing to a 57-prompt run. Set it to `False` only when you deliberately want the
full batch (and never for HunyuanVideo without a bigger GPU).

**T2V still on hold (2026-07-14 decision)**: finish all image tasks before running this for real - this notebook is
ready-to-go for when that happens, no further code changes needed.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
import os
from dotenv import load_dotenv
load_dotenv()

# Choose one: 'wan21' | 'hunyuanvideo' | 'cogvideox' | 'ltxvideo'
# wan21, cogvideox, ltxvideo are free/open-weight and fit an 8GB card. hunyuanvideo needs a cloud GPU (see markdown).
MODEL_KEY = 'wan21'

MODEL_MAP = {
    # Wan2.1 1.3B: the only Wan2.1 T2V checkpoint that plausibly fits an 8GB card. Do NOT swap in the -14B repo here.
    'wan21'        : ('Wan-AI/Wan2.1-T2V-1.3B-Diffusers', 'Wan2.1_T2V_1.3B'),
    # HunyuanVideo: 13B params, ~14GB VRAM even int4-quantized + offloaded. Will very likely OOM on 8GB - see markdown above.
    'hunyuanvideo' : ('hunyuanvideo-community/HunyuanVideo', 'HunyuanVideo'),
    # CogVideoX-2B: Apache 2.0, 2B params, fits comfortably on 8GB with offload+tiling.
    'cogvideox'    : ('zai-org/CogVideoX-2b', 'CogVideoX_2B'),
    # LTX-Video 0.9.7 distilled: research-use license, 2B distilled, very fast (4-10 steps), guidance_scale must be 1.0.
    'ltxvideo'     : ('Lightricks/LTX-Video-0.9.7-distilled', 'LTXVideo_0.9.7_distilled'),
}

HF_MODEL_ID, MODEL_LABEL = MODEL_MAP[MODEL_KEY]
HF_TOKEN = os.environ.get('HF_TOKEN')

# Per-model generation defaults (kept modest for local/8GB feasibility; raise for a real cloud run)
GEN_PARAMS = {
    'wan21': dict(
        height=480, width=832, num_frames=81, num_inference_steps=30,
        guidance_scale=5.0, flow_shift=3.0, fps=16,
    ),
    'hunyuanvideo': dict(
        height=320, width=512, num_frames=61, num_inference_steps=30,
        guidance_scale=6.0, fps=15,
    ),
    'cogvideox': dict(
        height=480, width=720, num_frames=49, num_inference_steps=50,
        guidance_scale=6.0, fps=8,
    ),
    'ltxvideo': dict(
        height=480, width=704, num_frames=121, num_inference_steps=8,
        guidance_scale=1.0, fps=24,   # distilled: guidance_scale MUST be 1.0, 4-10 steps
    ),
}[MODEL_KEY]

# Smoke-test guard - see markdown note. Flip to False only for a deliberate full run.
SMOKE_TEST        = True
SMOKE_TEST_FRAMES = 33   # short clip (4*8+1) for a quick end-to-end check
VIDEOS_PER_PROMPT = 1

REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROMPTS_XLS = os.path.join(REPO_ROOT, 'prompts', 'all_prompts.xlsx')
OUTPUT_DIR  = os.path.join(REPO_ROOT, 'dataset', 'media', 'videos', MODEL_LABEL)
INDEX_CSV   = os.path.join(REPO_ROOT, 'dataset', 'labels', f'generated_{MODEL_LABEL}.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(INDEX_CSV), exist_ok=True)

print(f'Model   : {HF_MODEL_ID}')
print(f'Label   : {MODEL_LABEL}')
print(f'Output  : {OUTPUT_DIR}')
print(f'Params  : {GEN_PARAMS}')
print(f'SMOKE_TEST: {SMOKE_TEST}')

In [ ]:
# ── GPU check ────────────────────────────────────────────────────────────────
# Before loading anything: if another process already has significant VRAM allocated
# (e.g. a sibling image-generation notebook running on the same machine), stop here —
# loading a video model on top of it will OOM both jobs.
import subprocess
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader,nounits'],
                          capture_output=True, text=True, timeout=10)
    used_mb, total_mb = [int(x.strip()) for x in out.stdout.strip().split(',')]
    print(f'GPU memory: {used_mb} MiB used / {total_mb} MiB total')
    if used_mb > 500:
        print('WARNING: significant GPU memory already in use by another process.')
        print('Do NOT proceed with loading/generation until it frees up — check nvidia-smi manually.')
except Exception as e:
    print(f'Could not query nvidia-smi ({e}); check GPU availability manually before proceeding.')

In [ ]:
# ── Load prompts ─────────────────────────────────────────────────────────────
import pandas as pd

df_all = pd.read_excel(PROMPTS_XLS, sheet_name='All Prompts (100)')
df_all.columns = df_all.columns.str.strip()

# Video-compatible prompts = Modality in {'Video', 'Both'}
df_vid = df_all[df_all['Modality'].isin(['Video', 'Both'])].copy().reset_index(drop=True)

if SMOKE_TEST:
    df_vid = df_vid.head(1).copy()
    print('SMOKE_TEST is on: restricting to 1 prompt.')

print(f'Total prompts    : {len(df_all)}')
print(f'Prompts this run : {len(df_vid)}')

In [ ]:
# ── Load model ───────────────────────────────────────────────────────
import torch, gc
from huggingface_hub import login

if HF_TOKEN:
    login(HF_TOKEN)

if MODEL_KEY == 'wan21':
    from diffusers import AutoencoderKLWan, WanPipeline
    from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

    vae = AutoencoderKLWan.from_pretrained(HF_MODEL_ID, subfolder='vae', torch_dtype=torch.float32, token=HF_TOKEN)
    pipe = WanPipeline.from_pretrained(HF_MODEL_ID, vae=vae, torch_dtype=torch.bfloat16, token=HF_TOKEN)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=GEN_PARAMS['flow_shift'])
    pipe.enable_model_cpu_offload()   # keeps the 1.3B model within 8GB
    pipe.vae.enable_tiling()

elif MODEL_KEY == 'hunyuanvideo':
    from diffusers import HunyuanVideoPipeline
    from diffusers.quantizers import PipelineQuantizationConfig

    # int4 quantization of the transformer - still ~14GB per the official docs, see markdown warning above.
    pipeline_quant_config = PipelineQuantizationConfig(
        quant_backend='bitsandbytes_4bit',
        quant_kwargs={'load_in_4bit': True, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_compute_dtype': torch.bfloat16},
        components_to_quantize=['transformer'],
    )
    pipe = HunyuanVideoPipeline.from_pretrained(
        HF_MODEL_ID, quantization_config=pipeline_quant_config, torch_dtype=torch.bfloat16, token=HF_TOKEN,
    )
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()

elif MODEL_KEY == 'cogvideox':
    from diffusers import CogVideoXPipeline

    pipe = CogVideoXPipeline.from_pretrained(HF_MODEL_ID, torch_dtype=torch.float16, token=HF_TOKEN)
    pipe.enable_model_cpu_offload()   # 2B fits within 8GB with offload; safety margin vs the 19GB figure quoted for 5B
    pipe.vae.enable_tiling()

elif MODEL_KEY == 'ltxvideo':
    from diffusers import LTXPipeline

    pipe = LTXPipeline.from_pretrained(HF_MODEL_ID, torch_dtype=torch.bfloat16, token=HF_TOKEN)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()

else:
    raise ValueError(f'Unknown MODEL_KEY: {MODEL_KEY}')

pipe.set_progress_bar_config(disable=True)
print(f'{MODEL_LABEL} loaded.')

In [ ]:
# ── Generation helpers ──────────────────────────────────────────────
import hashlib, datetime
from diffusers.utils import export_to_video

LOW_QUALITY_NEGATIVE = 'worst quality, inconsistent motion, blurry, jittery, distorted'


def make_filename(prompt_id, prompt_type, index=0):
    ts = datetime.datetime.now().strftime('%Y%m%d%H%M%S%f')
    h  = hashlib.sha256(f'{prompt_id}_{index}_{ts}'.encode()).hexdigest()[:10]
    t  = 'amb' if prompt_type == 'Ambiguous' else 'exp'
    return f'P{prompt_id:03d}_{t}_{h}.mp4'


def generate_video(prompt_text):
    num_frames = SMOKE_TEST_FRAMES if SMOKE_TEST else GEN_PARAMS['num_frames']
    with torch.no_grad():
        if MODEL_KEY == 'wan21':
            result = pipe(
                prompt=prompt_text,
                height=GEN_PARAMS['height'], width=GEN_PARAMS['width'],
                num_frames=num_frames,
                num_inference_steps=GEN_PARAMS['num_inference_steps'],
                guidance_scale=GEN_PARAMS['guidance_scale'],
            )
        elif MODEL_KEY == 'hunyuanvideo':
            result = pipe(
                prompt=prompt_text,
                height=GEN_PARAMS['height'], width=GEN_PARAMS['width'],
                num_frames=num_frames,
                num_inference_steps=GEN_PARAMS['num_inference_steps'],
                guidance_scale=GEN_PARAMS['guidance_scale'],
            )
        elif MODEL_KEY == 'cogvideox':
            result = pipe(
                prompt=prompt_text,
                height=GEN_PARAMS['height'], width=GEN_PARAMS['width'],
                num_frames=num_frames,
                num_inference_steps=GEN_PARAMS['num_inference_steps'],
                guidance_scale=GEN_PARAMS['guidance_scale'],
            )
        else:  # ltxvideo
            result = pipe(
                prompt=prompt_text,
                negative_prompt=LOW_QUALITY_NEGATIVE,
                height=GEN_PARAMS['height'], width=GEN_PARAMS['width'],
                num_frames=num_frames,
                num_inference_steps=GEN_PARAMS['num_inference_steps'],
                guidance_scale=GEN_PARAMS['guidance_scale'],
            )
    return result.frames[0]

print('Helpers ready.')

In [ ]:
# ── Resume: load already-generated index ──────────────────────────
# NOTE: dedup key includes prompt_type, not just prompt_id - numeric IDs repeat across
# Ambiguous/Explicit, and keying on prompt_id alone silently skips real prompts (found
# and fixed the same bug in generate_gemini_flash.ipynb on 2026-07-14).
if os.path.exists(INDEX_CSV):
    df_done  = pd.read_csv(INDEX_CSV)
    done_ids = set(zip(df_done['prompt_type'], df_done['prompt_id'], df_done['vid_index']))
    print(f'Resuming: {len(df_done)} videos already generated.')
else:
    df_done  = pd.DataFrame()
    done_ids = set()
    print('Starting fresh.')

In [ ]:
# ── Main generation loop ─────────────────────────────────────────────────────
from tqdm.notebook import tqdm

records = []

for _, row in tqdm(df_vid.iterrows(), total=len(df_vid), desc=MODEL_LABEL):
    pid         = int(row['ID'])
    ptype       = row['Type']
    prompt_text = str(row['Prompt Text'])

    for vid_idx in range(VIDEOS_PER_PROMPT):
        if (ptype, pid, vid_idx) in done_ids:
            continue

        status = 'ok'
        fname  = ''
        try:
            frames = generate_video(prompt_text)
            fname = make_filename(pid, ptype, vid_idx)
            export_to_video(frames, os.path.join(OUTPUT_DIR, fname), fps=GEN_PARAMS['fps'])
            done_ids.add((ptype, pid, vid_idx))
            torch.cuda.empty_cache()
            gc.collect()
        except torch.cuda.OutOfMemoryError as e:
            print(f'  OOM P{pid:03d}: {str(e)[:150]}')
            status = 'oom_error'
            torch.cuda.empty_cache()
            gc.collect()
        except Exception as e:
            print(f'  Error P{pid:03d}: {str(e)[:150]}')
            status = 'error'

        records.append({
            'file_path'   : f'videos/{MODEL_LABEL}/{fname}' if fname else '',
            'prompt_id'   : pid,
            'vid_index'   : vid_idx,
            'prompt_type' : ptype,
            'user_profile': row['User Profile'],
            'modality'    : row['Modality'],
            'category'    : row['Category'],
            'language'    : row['Language'],
            'status'      : status,
            'model'       : MODEL_LABEL,
            'prompt_text' : prompt_text,
        })

df_new = pd.DataFrame(records)
df_out = pd.concat([df_done, df_new], ignore_index=True) if not df_done.empty else df_new
df_out.to_csv(INDEX_CSV, index=False)

ok  = (df_out['status'] == 'ok').sum()
err = df_out['status'].isin(['error', 'oom_error']).sum()
print(f'\nDone. Generated: {ok} | Errors/OOM: {err}')
print(f'Index: {INDEX_CSV}')

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
df_results = pd.read_csv(INDEX_CSV)
print(f'=== {MODEL_LABEL} Generation Summary ===')
print(df_results.groupby(['prompt_type', 'status']).size().unstack(fill_value=0).to_string())
print(f"\nTotal videos saved: {(df_results['status']=='ok').sum()} / {len(df_results)}")